# EDA 

## 1. Configurations

In [7]:
import pandas as pd
import re
import os
from pathlib import Path
from dotenv import load_dotenv
from urllib.parse import parse_qsl, urlsplit
import requests

In [2]:
PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")


def require_secret(name: str) -> str:
    """Return a configured secret without printing or storing it in the notebook."""
    value = os.getenv(name)
    if not value:
        raise RuntimeError(
            f"{name} is not configured. Add it to the project-root .env file."
        )
    return value

## 2. Data Loader

### 2.1 Massive

- **Treasury Yields:** Daily frequency; 3-month, 2-year, and 10-year Treasury yields, with the 10-year–3-month spread derived as `T10Y3M`.
- **Inflation Expectations:** Monthly frequency; 10-year breakeven inflation, 5-year/5-year forward inflation, and Cleveland Fed model-based expectations.
- **Labor Market:** Monthly frequency; unemployment rate, labor-force participation, average hourly earnings, and job openings.
- **Funding Conditions:** Daily frequency; effective federal funds rate, SOFR, policy-rate bounds, repo activity, commercial-paper rates, and overnight funding volumes.
- **Realized Inflation:** Monthly frequency; headline and core CPI/PCE indexes, PCE spending, and derived realized month-over-month inflation rates.
- **Timing Note:** Monthly timestamps represent observation periods and will require release-date alignment before entering the weekly model.

In [4]:
# Paste after the configuration cell in Data_Engineering_v2.ipynb.

MASSIVE_ECONOMY_ENDPOINTS = {
    "treasury": "treasury-yields",
    "inflation": "inflation",
    "expectations": "inflation-expectations",
    "labor": "labor-market",
    "funding": "funding-conditions",
}


def load_massive_economy(
    endpoint: str,
    start_date: str | None = None,
    end_date: str | None = None,
) -> pd.DataFrame:
    if endpoint not in MASSIVE_ECONOMY_ENDPOINTS.values():
        raise ValueError(f"Unsupported Massive Economy endpoint: {endpoint}")

    api_key = require_secret("MASSIVE_API_KEY")
    url = f"https://api.massive.com/fed/v1/{endpoint}"

    params = {
        "apiKey": api_key,
        "limit": 50_000,
        "sort": "date.asc",
    }

    if start_date is not None:
        params["date.gte"] = start_date

    if end_date is not None:
        params["date.lte"] = end_date

    rows = []
    first_request = True

    with requests.Session() as session:
        while url:
            if first_request:
                request_params = params
                first_request = False
            else:
                query_keys = {
                    key for key, _ in parse_qsl(urlsplit(url).query)
                }
                request_params = (
                    None if "apiKey" in query_keys
                    else {"apiKey": api_key}
                )

            try:
                response = session.get(
                    url,
                    params=request_params,
                    timeout=30,
                )
            except requests.RequestException:
                raise RuntimeError(
                    f"Massive request failed for {endpoint}"
                ) from None

            if not response.ok:
                raise RuntimeError(
                    f"Massive returned HTTP {response.status_code} "
                    f"for {endpoint}"
                )

            try:
                payload = response.json()
            except requests.JSONDecodeError:
                raise RuntimeError(
                    f"Massive returned invalid JSON for {endpoint}"
                ) from None

            rows.extend(payload.get("results", []))
            url = payload.get("next_url")

    if not rows:
        return pd.DataFrame(index=pd.DatetimeIndex([], name="date"))

    frame = pd.DataFrame(rows)

    if "date" not in frame.columns:
        raise RuntimeError(
            f"Massive response for {endpoint} has no date field"
        )

    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")

    for column in frame.columns.difference(["date"]):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    return (
        frame.dropna(subset=["date"])
        .drop_duplicates(subset=["date"], keep="last")
        .set_index("date")
        .sort_index()
    )


START_DATE = "2014-01-01"
END_DATE = pd.Timestamp.today().normalize().strftime("%Y-%m-%d")

massive_treasury_raw = load_massive_economy(
    "treasury-yields",
    START_DATE,
    END_DATE,
)

massive_expectations_raw = load_massive_economy(
    "inflation-expectations",
    START_DATE,
    END_DATE,
)

massive_labor_raw = load_massive_economy(
    "labor-market",
    START_DATE,
    END_DATE,
)

massive_funding_raw = load_massive_economy(
    "funding-conditions",
    START_DATE,
    END_DATE,
)

massive_inflation_raw = load_massive_economy(
    "inflation",
    START_DATE,
    END_DATE,
)


massive_rates_daily = (
    massive_treasury_raw[
        ["yield_3_month", "yield_2_year", "yield_10_year"]
    ]
    .rename(
        columns={
            "yield_3_month": "ust_3m",
            "yield_2_year": "ust_2y",
            "yield_10_year": "ust_10y",
        }
    )
    .copy()
)

massive_rates_daily["T10Y3M"] = (
    massive_rates_daily["ust_10y"]
    - massive_rates_daily["ust_3m"]
)


massive_expectations_monthly = (
    massive_expectations_raw.rename(
        columns={
            "market_10_year": "T10YIE",
            "forward_years_5_to_10": "infl_5y5y",
        }
    )
    .copy()
)


massive_labor_monthly = (
    massive_labor_raw.rename(
        columns={
            "unemployment_rate": "unrate",
        }
    )
    .copy()
)


massive_funding_daily = (
    massive_funding_raw.rename(
        columns={
            "effective_fed_funds_rate": "fedfunds",
            "secured_overnight_financing_rate": "SOFR",
        }
    )
    .copy()
)


massive_inflation_monthly = (
    massive_inflation_raw.rename(
        columns={
            "cpi": "cpi_index",
            "cpi_core": "core_cpi_index",
            "pce": "pce_index",
            "pce_core": "core_pce_index",
        }
    )
    .copy()
)

massive_inflation_monthly["cpi_mom_realized"] = (
    massive_inflation_monthly["cpi_index"]
    .pct_change(fill_method=None)
    .mul(100)
)

massive_inflation_monthly["core_cpi_mom_realized"] = (
    massive_inflation_monthly["core_cpi_index"]
    .pct_change(fill_method=None)
    .mul(100)
)

massive_inflation_monthly["pce_mom_realized"] = (
    massive_inflation_monthly["pce_index"]
    .pct_change(fill_method=None)
    .mul(100)
)

massive_inflation_monthly["core_pce_mom_realized"] = (
    massive_inflation_monthly["core_pce_index"]
    .pct_change(fill_method=None)
    .mul(100)
)


massive_datasets = {
    "treasury_daily": massive_rates_daily,
    "expectations_monthly": massive_expectations_monthly,
    "labor_monthly": massive_labor_monthly,
    "funding_daily": massive_funding_daily,
    "inflation_monthly": massive_inflation_monthly,
}

massive_coverage = pd.DataFrame(
    {
        name: {
            "rows": len(frame),
            "start": frame.index.min(),
            "end": frame.index.max(),
            "columns": ", ".join(frame.columns),
        }
        for name, frame in massive_datasets.items()
    }
).T

massive_coverage

,rows,start,end,columns
treasury_daily,3171,2014-01-02 00:00:00,2026-09-04 00:00:00,"ust_3m, ust_2y, ust_10y, T10Y3M"
expectations_monthly,152,2014-01-01 00:00:00,2026-08-01 00:00:00,"market_5_year, T10YIE, infl_5y5y, model_1_year..."
labor_monthly,152,2014-01-01 00:00:00,2026-08-01 00:00:00,"unrate, labor_force_participation_rate, avg_ho..."
funding_daily,4635,2014-01-01 00:00:00,2026-09-09 00:00:00,"fedfunds, fed_funds_target_upper, fed_funds_ta..."
inflation_monthly,151,2014-01-01 00:00:00,2026-07-01 00:00:00,"cpi_index, cpi_year_over_year, core_cpi_index,..."


### 2.2 FRED

- **Initial Unemployment Claims (`ICSA`):** Weekly frequency; high-frequency indicator of labor-market deterioration.
- **Unemployment Rate (`UNRATE`):** Monthly frequency; retained for point-in-time comparison with Massive labor data.
- **Real-Time Sahm Rule (`SAHMREALTIME`):** Monthly frequency; recession indicator based on changes in unemployment.
- **10-Year Breakeven Inflation (`T10YIE`):** Daily frequency; market-implied long-term inflation expectations.
- **10-Year Real Yield (`DFII10`):** Daily frequency; inflation-adjusted Treasury yield and measure of real-rate pressure.
- **5-Year/5-Year Forward Inflation (`T5YIFR`):** Daily frequency; long-term forward inflation expectations.
- **Financial Conditions (`NFCI`, `ANFCI`):** Weekly frequency; broad and adjusted measures of U.S. financial conditions.
- **High-Yield Credit Spread (`BAMLH0A0HYM2`):** Daily frequency; option-adjusted spread for U.S. high-yield corporate bonds.
- **Economic Policy Uncertainty (`USEPUINDXD`):** Daily frequency; news-based measure of U.S. policy uncertainty.
- **WTI Crude Oil (`DCOILWTICO`):** Daily frequency; spot oil price used in the inflation-shock bucket.
- **Broad U.S. Dollar Index (`DTWEXBGS`):** Daily frequency; trade-weighted dollar measure used in the financial-stress bucket.
- **Vintage Note:** The current dataset is loaded as an ALFRED/FRED snapshot using the configured `FRED_VINTAGE_DATE`.

In [5]:
FRED_SERIES = {
    # Growth and labor
    "ICSA": "ICSA",
    "unrate": "UNRATE",
    "sahm_realtime": "SAHMREALTIME",

    # Rates and inflation expectations
    # Retained because FRED provides higher-frequency observations than Massive.
    "T10YIE": "T10YIE",
    "DFII10": "DFII10",
    "infl_5y5y": "T5YIFR",

    # Financial conditions and stress
    "NFCI": "NFCI",
    "ANFCI": "ANFCI",
    "HY_OAS": "BAMLH0A0HYM2",
    "EPU": "USEPUINDXD",

    # Commodity and currency
    "oil_wti": "DCOILWTICO",
    "usd_broad": "DTWEXBGS",
}


def load_fred_series(
    series_id: str,
    start_date: str | None = None,
    end_date: str | None = None,
    vintage_date: str | None = None,
) -> pd.Series:
    api_key = require_secret("FRED_API_KEY")
    url = "https://api.stlouisfed.org/fred/series/observations"

    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "sort_order": "asc",
        "limit": 100_000,
        "offset": 0,
    }

    if start_date is not None:
        params["observation_start"] = start_date

    if end_date is not None:
        params["observation_end"] = end_date

    if vintage_date is not None:
        params["realtime_start"] = vintage_date
        params["realtime_end"] = vintage_date

    observations = []

    with requests.Session() as session:
        while True:
            try:
                response = session.get(
                    url,
                    params=params,
                    timeout=30,
                )
            except requests.RequestException:
                raise RuntimeError(
                    f"FRED request failed for {series_id}"
                ) from None

            if not response.ok:
                raise RuntimeError(
                    f"FRED returned HTTP {response.status_code} "
                    f"for {series_id}"
                )

            try:
                payload = response.json()
            except requests.JSONDecodeError:
                raise RuntimeError(
                    f"FRED returned invalid JSON for {series_id}"
                ) from None

            batch = payload.get("observations", [])
            observations.extend(batch)

            count = int(payload.get("count", len(observations)))
            offset = int(payload.get("offset", params["offset"]))
            limit = int(payload.get("limit", params["limit"]))

            if offset + limit >= count:
                break

            params["offset"] = offset + limit

    if not observations:
        return pd.Series(
            dtype="float64",
            name=series_id,
            index=pd.DatetimeIndex([], name="date"),
        )

    frame = pd.DataFrame(observations)

    frame["date"] = pd.to_datetime(
        frame["date"],
        errors="coerce",
    )

    frame["value"] = pd.to_numeric(
        frame["value"],
        errors="coerce",
    )

    series = (
        frame.dropna(subset=["date"])
        .drop_duplicates(subset=["date"], keep="last")
        .set_index("date")["value"]
        .sort_index()
    )

    series.name = series_id
    return series


def load_fred_panel(
    series_map: dict[str, str],
    start_date: str | None = None,
    end_date: str | None = None,
    vintage_date: str | None = None,
) -> pd.DataFrame:
    series = {
        model_name: load_fred_series(
            series_id=series_id,
            start_date=start_date,
            end_date=end_date,
            vintage_date=vintage_date,
        )
        for model_name, series_id in series_map.items()
    }

    panel = pd.concat(series, axis=1).sort_index()
    panel.index.name = "date"

    return panel


FRED_START_DATE = "2014-01-01"
FRED_END_DATE = pd.Timestamp.today().normalize().strftime("%Y-%m-%d")
FRED_VINTAGE_DATE = FRED_END_DATE

fred_raw = load_fred_panel(
    series_map=FRED_SERIES,
    start_date=FRED_START_DATE,
    end_date=FRED_END_DATE,
    vintage_date=FRED_VINTAGE_DATE,
)


fred_coverage = pd.DataFrame(
    {
        column: {
            "fred_series": FRED_SERIES[column],
            "observations": fred_raw[column].notna().sum(),
            "first_valid": fred_raw[column].first_valid_index(),
            "last_valid": fred_raw[column].last_valid_index(),
        }
        for column in fred_raw.columns
    }
).T

fred_coverage

,fred_series,observations,first_valid,last_valid
ICSA,ICSA,661,2014-01-04 00:00:00,2026-08-29 00:00:00
unrate,UNRATE,151,2014-01-01 00:00:00,2026-08-01 00:00:00
sahm_realtime,SAHMREALTIME,151,2014-01-01 00:00:00,2026-08-01 00:00:00
T10YIE,T10YIE,3172,2014-01-02 00:00:00,2026-09-08 00:00:00
DFII10,DFII10,3171,2014-01-02 00:00:00,2026-09-04 00:00:00
infl_5y5y,T5YIFR,3172,2014-01-02 00:00:00,2026-09-08 00:00:00
NFCI,NFCI,661,2014-01-03 00:00:00,2026-08-28 00:00:00
ANFCI,ANFCI,661,2014-01-03 00:00:00,2026-08-28 00:00:00
HY_OAS,BAMLH0A0HYM2,786,2023-09-11 00:00:00,2026-09-08 00:00:00
EPU,USEPUINDXD,4634,2014-01-01 00:00:00,2026-09-08 00:00:00


### 2.3 CBOE VIX

In [6]:
CBOE_VIX_URL = (
    "https://cdn.cboe.com/api/global/us_indices/"
    "daily_prices/VIX_History.csv"
)

vix_raw = pd.read_csv(CBOE_VIX_URL, parse_dates=["DATE"])

vix = (
    vix_raw
    .rename(columns={"DATE": "date", "CLOSE": "VIX"})
    .set_index("date")["VIX"]
    .sort_index()
)

### 2.4 Cleveland Inflation Nowcasts

In [8]:
INFLATION_NOWCAST_DIR = (
    Path.home()
    / "Desktop"
    / "Data"
    / "Macro"
    / "Inflation_Nowcast"
)

INFLATION_NOWCAST_COLUMNS = {
    "CPI Inflation": "cpi_inflation_mom",
    "Core CPI Inflation": "core_cpi_inflation_mom",
    "PCE Inflation": "pce_inflation_mom",
    "Core PCE Inflation": "core_pce_inflation_mom",
}

INFLATION_VALUE_COLUMNS = list(
    INFLATION_NOWCAST_COLUMNS.values()
)


def parse_target_month(file_path: Path) -> pd.Timestamp:
    pattern = (
        r"Month-Over-MonthPercentChange-"
        r"(\d{4})-(\d{1,2})\.csv"
    )

    match = re.fullmatch(pattern, file_path.name)

    if match is None:
        raise ValueError(
            f"Invalid inflation-nowcast filename: {file_path.name}"
        )

    year = int(match.group(1))
    month = int(match.group(2))

    return pd.Timestamp(year=year, month=month, day=1)


def parse_forecast_dates(
    labels: pd.Series,
    starting_year: int,
) -> pd.DatetimeIndex:
    dates = []
    current_year = starting_year
    previous_month = None

    for label in labels:
        month, day = map(
            int,
            str(label).strip().split("/"),
        )

        if (
            previous_month is not None
            and month < previous_month
        ):
            current_year += 1

        dates.append(
            pd.Timestamp(
                year=current_year,
                month=month,
                day=day,
            )
        )

        previous_month = month

    index = pd.DatetimeIndex(dates)

    if not index.is_monotonic_increasing:
        raise ValueError(
            "Forecast dates are not chronologically ordered."
        )

    return index


def read_inflation_nowcast_file(
    file_path: Path,
) -> pd.DataFrame:
    target_month = parse_target_month(file_path)

    frame = pd.read_csv(file_path)
    frame.columns = frame.columns.str.strip()

    required_columns = [
        "Label",
        *INFLATION_NOWCAST_COLUMNS,
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in frame.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{file_path.name} is missing columns: "
            f"{missing_columns}"
        )

    frame = frame[required_columns].rename(
        columns=INFLATION_NOWCAST_COLUMNS
    )

    frame["forecast_date"] = parse_forecast_dates(
        frame["Label"],
        starting_year=target_month.year,
    )

    frame["target_month"] = target_month

    for column in INFLATION_VALUE_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    return frame[
        [
            "forecast_date",
            "target_month",
            *INFLATION_VALUE_COLUMNS,
        ]
    ]


def load_inflation_nowcasts(
    folder: Path,
) -> pd.DataFrame:
    files = list(
        folder.glob(
            "Month-Over-MonthPercentChange-*.csv"
        )
    )

    if not files:
        raise FileNotFoundError(
            f"No inflation-nowcast files found in {folder}"
        )

    files = sorted(
        files,
        key=lambda path: parse_target_month(path),
    )

    frames = [
        read_inflation_nowcast_file(file_path)
        for file_path in files
    ]

    combined = (
        pd.concat(frames, ignore_index=True)
        .sort_values(
            ["forecast_date", "target_month"]
        )
    )

    duplicate_pairs = combined.duplicated(
        ["forecast_date", "target_month"]
    )

    if duplicate_pairs.any():
        raise ValueError(
            "Duplicate forecast-date and target-month pairs found."
        )

    return combined.set_index(
        ["forecast_date", "target_month"]
    ).sort_index()


def select_front_inflation_nowcast(
    nowcast_long: pd.DataFrame,
) -> pd.DataFrame:
    """
    For each forecast date and inflation measure, select the
    earliest target month that still has a published nowcast.
    """

    frame = (
        nowcast_long.reset_index()
        .sort_values(
            ["forecast_date", "target_month"]
        )
    )

    selected = {}

    for column in INFLATION_VALUE_COLUMNS:
        selected[column] = (
            frame.dropna(subset=[column])
            .groupby("forecast_date", sort=True)[column]
            .first()
        )

    output = pd.DataFrame(selected).sort_index()
    output.index.name = "forecast_date"

    return output


inflation_nowcast_long = load_inflation_nowcasts(
    INFLATION_NOWCAST_DIR
)

infl_nowcast_daily = select_front_inflation_nowcast(
    inflation_nowcast_long
)


inflation_nowcast_coverage = pd.Series(
    {
        "files_loaded": (
            inflation_nowcast_long
            .index
            .get_level_values("target_month")
            .nunique()
        ),
        "long_rows": len(inflation_nowcast_long),
        "daily_rows": len(infl_nowcast_daily),
        "first_forecast_date": (
            infl_nowcast_daily.index.min()
        ),
        "last_forecast_date": (
            infl_nowcast_daily.index.max()
        ),
        "first_target_month": (
            inflation_nowcast_long
            .index
            .get_level_values("target_month")
            .min()
        ),
        "last_target_month": (
            inflation_nowcast_long
            .index
            .get_level_values("target_month")
            .max()
        ),
    },
    name="Cleveland Fed inflation nowcasts",
)

display(inflation_nowcast_coverage)
display(infl_nowcast_daily.tail())

files_loaded                           159
long_rows                             6567
daily_rows                            3276
first_forecast_date    2013-08-20 00:00:00
last_forecast_date     2026-09-09 00:00:00
first_target_month     2013-07-01 00:00:00
last_target_month      2026-09-01 00:00:00
Name: Cleveland Fed inflation nowcasts, dtype: object

,cpi_inflation_mom,core_cpi_inflation_mom,pce_inflation_mom,core_pce_inflation_mom
forecast_date,,,,
2026-09-02,0.359181,0.203342,0.353657,0.274535
2026-09-03,0.359181,0.203342,0.353657,0.274535
2026-09-04,0.359181,0.203342,0.353657,0.274535
2026-09-08,0.359181,0.203342,0.353657,0.274535
2026-09-09,0.359181,0.203342,0.353657,0.274535
